# 🎤 domovina.tv Speaker Embedding Backfill (Colab G4)

Ekstrahira **per-speaker voice embeddings** iz dijariziranih WAV-ova na Google Drive-u
(`MyDrive/domovina_fetch_data/canary_wav/`).

Model: **NVIDIA NeMo TitaNet-Large** (`nvidia/speakerverification_en_titanet_large`) — ~0.7% EER na VoxCeleb1,
192-dim L2-normalized embedding. Pokriva multilingual govor (uključujući hrvatski).

Output: `*.canary.diarized.embeddings.json` pored postojećih SRT/WAV datoteka. Format definiran u
[`docs/data_contract.md` §7](../docs/data_contract.md#7-reserved-canarydiarizedembeddingsjson-v11-planning).

**Idempotentno:** datoteke s postojećim `.embeddings.json` se preskaču — možeš pokretati više puta sigurno.

**Trajanje:** ~3-5h za 2 559 epizoda na G4 (RTX PRO 6000 Blackwell), ~$0.50-1.00 u Pro+ compute units.

---

## Preduvjeti

1. **Colab Pro+ s G4 GPU** (T4/L4 rade ali sporije)
2. **Google Drive mountan** s `MyDrive/domovina_fetch_data/canary_wav/` strukturom
3. WAV + `.canary.diarized.srt` datoteke već uploadane preko rclone (pipeline ih već pravi)

## 1️⃣ Mount Drive + provjeri GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print('\n=== GPU ===')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'],
                     capture_output=True, text=True).stdout)

import os
INPUT_DIR = '/content/drive/MyDrive/domovina_fetch_data/canary_wav'
assert os.path.isdir(INPUT_DIR), f'Drive direktorij nedostaje: {INPUT_DIR}'
print(f'\n✅ Drive mountan, input: {INPUT_DIR}')
print(f'   📊 Channel direktorija: {len(os.listdir(INPUT_DIR))}')

## 2️⃣ Instaliraj dependencies (NeMo + audio libs)

Prvi pokret ~2-3 min. Nakon toga cached u Colab runtime-u dok ga ne restartaš.

In [ ]:
!pip install -q 'nemo_toolkit[asr]==2.0.0' soundfile librosa
print('✅ Dependencies installed')

## 3️⃣ Dohvati workhorse skriptu iz Github-a

Vučemo direktno iz `domovinatv/fetch.domovina.tv` repa (public). Tako uvijek vrtimo zadnju verziju
bez ručnog uploadanja.

In [ ]:
!wget -q -O extract_speaker_embeddings.py https://raw.githubusercontent.com/domovinatv/fetch.domovina.tv/main/colab_speaker_embeddings/extract_speaker_embeddings.py
!ls -la extract_speaker_embeddings.py
!head -30 extract_speaker_embeddings.py

## 4️⃣ Smoke test — prvih 3 epizode, dry-run pregled

Provjeri da scan pronalazi prave datoteke.

In [ ]:
!python extract_speaker_embeddings.py --input-dir "$INPUT_DIR" --source canary --limit 5 --dry-run


## 5️⃣ Smoke test — stvarno procesiranje prvih 3 epizoda

Provjeri da model loadanje, embedding extraction i JSON spremanje rade end-to-end. Trebalo bi proći
u ~30-60 sekundi za 3 epizode.

In [ ]:
!python extract_speaker_embeddings.py --input-dir "$INPUT_DIR" --source canary --limit 3

## 6️⃣ Pun backfill — sve dostupne epizode

Idempotentno (svaki re-run preskače već obrađene). Ako prekineš (Colab session timeout, manual stop),
sljedeći run nastavlja od mjesta gdje je stalo.

**`--max-runtime-hours 11`** osigurava da skripta sama izađe prije Colab Pro+ 12h limita
(stiše čistim spremanjem stanja).

In [ ]:
!python extract_speaker_embeddings.py --input-dir "$INPUT_DIR" --source canary --max-runtime-hours 11

## 7️⃣ (Opcionalno) Sortformer paralelni backfill

Ako želiš embeddinge i za eksperimentalni `*.sortformer.diarized.srt` outpute (radi ensemble-a u
domovina-rag importu, vidi [§15.10 plana](../docs/rag_clickhouse_postgres_plan.md#1510-pyannote-failure-modes-i-sortformer-ensemble-strategija)),
ponovi s `--source sortformer`.

In [ ]:
!python extract_speaker_embeddings.py --input-dir "$INPUT_DIR" --source sortformer --max-runtime-hours 4

## 8️⃣ Verifikacija — koliko embedding fajlova je generirano

In [ ]:
import subprocess
print('=== Canary embeddings ===')
out = subprocess.run(['find', INPUT_DIR, '-name', '*.canary.diarized.embeddings.json'],
                     capture_output=True, text=True)
canary_files = out.stdout.strip().split('\n')
canary_files = [f for f in canary_files if f]
print(f'Ukupno: {len(canary_files)}')

print('\n=== Sortformer embeddings ===')
out = subprocess.run(['find', INPUT_DIR, '-name', '*.sortformer.diarized.embeddings.json'],
                     capture_output=True, text=True)
sort_files = out.stdout.strip().split('\n')
sort_files = [f for f in sort_files if f]
print(f'Ukupno: {len(sort_files)}')

## 9️⃣ Sync nazad na Mac (opcionalno)

Embedding JSON datoteke su sada na Drive-u. Sljedeći `run_pipeline.sh` automatski ih povuče
u Korak 0 (rclone filter `+ **.embeddings.*.json` već postavljen). Manual sync:

```bash
rclone copy google_drive_ms:domovina_fetch_data/canary_wav $OUTPUT_DIR \
    -L --filter '+ **.embeddings.*.json' --filter '- *' \
    --drive-shared-with-me --progress
```

Filter `**.embeddings.*.json` match-a sve modele (`titanet`, `pyannote_wespeaker34`, ...) — pattern
podržan u `run_pipeline.sh` korak 0 (auto sync sa svakim novim pokretanjem).